# Qwen3.5 CPU MWE

**MWE** bedeutet **Minimal Working Example**: das kleinstmoegliche lauffaehige Beispiel fuer genau den relevanten Fall.

## Ziel

Dieses Notebook zeigt zwei minimale CPU-Pfade fuer `Qwen/Qwen3.5-0.8B` im Env `qwen35_stable`:

- **Transformers** als direkter Python-Default
- **GGUF + llama-server** als externer Runtime-Pfad

Beide Pfade nutzen denselben Labeling-Prompt und koennen danach direkt auf Geschwindigkeit verglichen werden.

## Entscheidung

`Transformers` bleibt hier der einfachste Python-Default.

`llama-server` ist die minimale GGUF-Erweiterung fuer diesen Test, weil Python dabei nur einen lokalen Server anspricht und nicht selbst die native GGUF-Runtime einbettet.

## Benchmark-Ziel

Der Vergleich unten misst **Warm-Run-Inferenzzeit** fuer denselben Prompt. Der GGUF-Benchmark macht dafuer einen untimed Warmup und beendet nur einen vom Notebook selbst gestarteten Server danach automatisch.


In [1]:
import json
import statistics
import subprocess
import time
import urllib.request
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-0.8B"
CPU_THREADS = 4
LABEL_KWARGS = {"max_new_tokens": 8, "temperature": 0.0}

torch.set_num_threads(CPU_THREADS)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="cpu",
)
model.eval()
model.generation_config.pad_token_id = tokenizer.pad_token_id

SAMPLE_KEYWORDS = "telescope, orbit, spacecraft, galaxy, mission"
SAMPLE_DOCS = "observations of distant galaxies, orbital dynamics, and deep-space missions"

def build_messages(keywords: str, representative_docs: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": "You label research topics with 1 to 3 words only. Output only the final label.",
        },
        {
            "role": "user",
            "content": (
                "Label this topic in 1-3 words only. "
                f"Topic keywords: {keywords}. "
                f"Representative docs: {representative_docs}."
            ),
        },
    ]

def render_prompt(keywords: str, representative_docs: str) -> str:
    return tokenizer.apply_chat_template(
        build_messages(keywords, representative_docs),
        tokenize=False,
        add_generation_prompt=True,
    )

def strip_think_tags(text: str) -> str:
    return text.replace("<think>", "").split("</think>", 1)[-1].strip()

def label_topic_transformers(
    keywords: str,
    representative_docs: str,
    max_new_tokens: int = 8,
    temperature: float = 0.0,
) -> dict:
    inputs = tokenizer(
        [render_prompt(keywords, representative_docs)],
        add_special_tokens=False,
        return_tensors="pt",
    )
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0.0,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if temperature > 0.0:
        generation_kwargs["temperature"] = temperature

    t0 = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(**inputs, **generation_kwargs)
    t1 = time.perf_counter()

    generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], outputs)
    ]
    label = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return {
        "backend": "transformers",
        "label": strip_think_tags(label),
        "input_tokens": int(inputs["input_ids"].shape[-1]),
        "generate_s": round(t1 - t0, 2),
    }

hf_sample = label_topic_transformers(SAMPLE_KEYWORDS, SAMPLE_DOCS, **LABEL_KWARGS)

print("backend:", hf_sample["backend"])
print("model_id:", MODEL_ID)
print("cpu_threads:", CPU_THREADS)
print("label:", hf_sample["label"])
print("input_tokens:", hf_sample["input_tokens"])
print("generate_s:", hf_sample["generate_s"])


c:\Users\rapha\anaconda3\envs\qwen35_stable\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 2839.95it/s]


backend: transformers
model_id: Qwen/Qwen3.5-0.8B
cpu_threads: 4
label: spacecraft
input_tokens: 76
generate_s: 15.1


## GGUF + llama-server

Dieser zweite Pfad bleibt ebenfalls minimal: Python sendet denselben Prompt an einen lokal laufenden `llama-server`.

Die Zelle darunter versucht zuerst `llama-server` automatisch ueber `PATH` zu finden. Fuer die GGUF-Datei ist unten bereits ein lokaler Testpfad eingetragen.

Wenn das Notebook den Server selbst startet, wird er nach Sample- und Benchmark-Lauf wieder beendet. Ein extern bereits laufender `llama-server` bleibt unberuehrt.


In [2]:
import atexit
import shutil

LLAMA_SERVER_EXE = Path(shutil.which("llama-server") or r"C:\\path\\to\\llama-server.exe")
GGUF_MODEL_PATH = Path(r"data\\models\\qwen35_gguf\\Qwen_Qwen3.5-0.8B-Q4_K_M.gguf")
LLAMA_PORT = 8001

llama_server_process = globals().get("llama_server_process")

def _post_json(url: str, payload: dict, timeout: float = 120.0) -> dict:
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def llama_server_issues() -> list[str]:
    issues = []
    if not LLAMA_SERVER_EXE.exists():
        issues.append(
            "llama-server not found. Put llama-server on PATH or set LLAMA_SERVER_EXE explicitly."
        )
    if not GGUF_MODEL_PATH.exists():
        issues.append(f"GGUF model not found: {GGUF_MODEL_PATH}")
    return issues

def llama_server_ready(port: int = LLAMA_PORT) -> bool:
    try:
        _post_json(
            f"http://127.0.0.1:{port}/completion",
            {"prompt": "ping", "n_predict": 1, "temperature": 0.0},
            timeout=5.0,
        )
        return True
    except Exception:
        return False

def ensure_llama_server(
    port: int = LLAMA_PORT,
    cpu_threads: int = CPU_THREADS,
    timeout_s: float = 120.0,
) -> None:
    global llama_server_process

    issues = llama_server_issues()
    if issues:
        raise FileNotFoundError("\n".join(issues))

    if llama_server_ready(port):
        return

    llama_server_process = subprocess.Popen(
        [
            str(LLAMA_SERVER_EXE),
            "-m",
            str(GGUF_MODEL_PATH),
            "--host",
            "127.0.0.1",
            "--port",
            str(port),
            "--threads",
            str(cpu_threads),
            "--ctx-size",
            "4096",
            "--reasoning",
            "off",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        creationflags=getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0),
    )

    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if llama_server_process.poll() is not None:
            raise RuntimeError("llama-server exited before becoming ready")
        if llama_server_ready(port):
            return
        time.sleep(1.0)

    raise TimeoutError("llama-server did not become ready in time")

def stop_llama_server() -> None:
    global llama_server_process

    if llama_server_process is None:
        return
    if llama_server_process.poll() is None:
        llama_server_process.terminate()
        try:
            llama_server_process.wait(timeout=10.0)
        except subprocess.TimeoutExpired:
            llama_server_process.kill()
            llama_server_process.wait(timeout=10.0)
    llama_server_process = None

atexit.register(stop_llama_server)

def label_topic_llama_server(
    keywords: str,
    representative_docs: str,
    max_new_tokens: int = 8,
    temperature: float = 0.0,
    port: int = LLAMA_PORT,
) -> dict:
    ensure_llama_server(port=port)

    t0 = time.perf_counter()
    response = _post_json(
        f"http://127.0.0.1:{port}/completion",
        {
            "prompt": render_prompt(keywords, representative_docs),
            "n_predict": max_new_tokens,
            "temperature": temperature,
        },
    )
    t1 = time.perf_counter()

    return {
        "backend": "llama_server",
        "label": strip_think_tags(response.get("content", "")),
        "input_tokens": int(response.get("tokens_evaluated", -1)),
        "generate_s": round(t1 - t0, 2),
    }

print("llama-server exe:", LLAMA_SERVER_EXE)
print("gguf_model_path:", GGUF_MODEL_PATH)
print("llama_port:", LLAMA_PORT)

llama_issues = llama_server_issues()
if llama_issues:
    print("llama_server sample skipped:")
    for issue in llama_issues:
        print(f"- {issue}")
else:
    try:
        llama_sample = label_topic_llama_server(SAMPLE_KEYWORDS, SAMPLE_DOCS, **LABEL_KWARGS)
        print("backend:", llama_sample["backend"])
        print("port:", LLAMA_PORT)
        print("label:", llama_sample["label"])
        print("input_tokens:", llama_sample["input_tokens"])
        print("generate_s:", llama_sample["generate_s"])
    finally:
        stop_llama_server()


llama-server exe: C:\Users\rapha\AppData\Local\Microsoft\WinGet\Packages\ggml.llamacpp_Microsoft.Winget.Source_8wekyb3d8bbwe\llama-server.EXE
gguf_model_path: data\models\qwen35_gguf\Qwen_Qwen3.5-0.8B-Q4_K_M.gguf
llama_port: 8001
backend: llama_server
port: 8001
label: spacecraft
input_tokens: 76
generate_s: 0.71


In [3]:
BENCHMARK_REPEATS = 5

def benchmark_backend(
    label_fn,
    backend_name: str,
    repeats: int = BENCHMARK_REPEATS,
    warmup: bool = False,
) -> dict:
    if warmup:
        label_fn(SAMPLE_KEYWORDS, SAMPLE_DOCS, **LABEL_KWARGS)

    runs = []
    last_result = None
    for _ in range(repeats):
        last_result = label_fn(SAMPLE_KEYWORDS, SAMPLE_DOCS, **LABEL_KWARGS)
        runs.append(last_result["generate_s"])

    return {
        "backend": backend_name,
        "label": last_result["label"],
        "input_tokens": last_result["input_tokens"],
        "best_s": round(min(runs), 2),
        "mean_s": round(statistics.mean(runs), 2),
        "runs_s": runs,
    }

results = [benchmark_backend(label_topic_transformers, "transformers", warmup=True)]

llama_issues = llama_server_issues()
if llama_issues:
    print("llama_server benchmark skipped:")
    for issue in llama_issues:
        print(f"- {issue}")
else:
    try:
        results.append(
            benchmark_backend(label_topic_llama_server, "llama_server", warmup=True)
        )
    finally:
        stop_llama_server()

print()
print("Benchmark warm runs")
print("-------------------")
for result in results:
    print(f"backend: {result['backend']}")
    print(f"label: {result['label']}")
    print(f"input_tokens: {result['input_tokens']}")
    print(f"best_s: {result['best_s']}")
    print(f"mean_s: {result['mean_s']}")
    print(f"runs_s: {result['runs_s']}")
    print()

if len(results) == 2:
    delta = round(results[0]["mean_s"] - results[1]["mean_s"], 2)
    print(f"mean_s delta (transformers - llama_server): {delta}")

print("llama_server notebook process stopped:", llama_server_process is None)



Benchmark warm runs
-------------------
backend: transformers
label: spacecraft
input_tokens: 76
best_s: 15.27
mean_s: 15.88
runs_s: [15.27, 16.27, 15.46, 16.36, 16.03]

backend: llama_server
label: spacecraft
input_tokens: 76
best_s: 0.19
mean_s: 0.21
runs_s: [0.21, 0.21, 0.19, 0.23, 0.2]

mean_s delta (transformers - llama_server): 15.67
llama_server notebook process stopped: True
